In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Install required packages
!pip install -q transformers accelerate bitsandbytes torch torchvision pillow matplotlib gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.1 MB/s eta 0:00:00


In [ ]:
# LLaVA 1.5-7B - Good for T4 GPU with 4-bit quantization
import torch
from transformers import LlavaProcessor, LlavaForConditionalGeneration
from PIL import Image
import requests
import matplotlib.pyplot as plt

# Model ID - smaller model that works on T4 GPU
model_id = "llava-hf/llava-1.5-7b-hf"

# Load processor
processor = LlavaProcessor.from_pretrained(model_id)

# Load model with 4-bit quantization to fit in T4 GPU memory
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)

print(" LLaVA model loaded successfully!")

def ask_llava(image, question):
    """
    Ask LLaVA a question about an image
    """
    # Format prompt correctly for LLaVA
    prompt = f"USER: <tr>\n{question}\nASSISTANT:"

    # Process inputs
    inputs = processor(prompt, images=image, return_tensors="pt").to(model.device)

    # Generate response
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.7
        )

    # Decode response
    response = processor.decode(output_ids[0], skip_special_tokens=True)

    # Extract only the assistant's response
    if "ASSISTANT:" in response:
        response = response.split("ASSISTANT:")[-1].strip()

    return response

# Test with a sample image
def test_llava():
    # Load a sample image
    url = "/content/drive/MyDrive/SEM-8 MAJOR PROJECT/Images/0/0_original_43.png_592b345c-34ee-4cc5-b2f4-774db9375ab8.png"
    image = Image.open(requests.get(url, stream=True).raw).convert('RGB')

    # Ask about traffic signs
    question = "What traffic sign do you see in this image? Describe it."

    response = ask_llava(image, question)

    # Display
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(image)
    plt.title("Image")
    plt.axis('off')

    plt.subplot(1, 2, 2)
    plt.text(0.1, 0.5, f"Q: {question}\n\nA: {response}",
             fontsize=12, wrap=True)
    plt.axis('off')
    plt.title("LLaVA Response")
    plt.tight_layout()
    plt.show()

    print(f"Question: {question}")
    print(f"Answer: {response}")

# test_llava()

processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

... (5 outputs left out)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 86.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 67.81 MiB is free. Including non-PyTorch memory, this process has 14.49 GiB memory in use. Of the allocated memory 14.06 GiB is allocated by PyTorch, and 307.52 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)